# 03 · Run the experiments

Runs the whole grid and writes one row per (seed, regime, model, condition, recalibration) to
`results/results.csv`.

- **Regime A (deployment degradation):** train on complete data, degrade validation/test only.
- **Regime B (adaptive):** train, validation and test all degraded the same way.
- **Recalibration:** Platt / isotonic, fitted on the degraded validation set only.

**Runtime.** The quick configuration (13 conditions, 2 seeds) serves as a smoke test and took about 7.5 minutes
(roughly 3.7 minutes per seed) on the development machine. The full grid (32 conditions, 5 seeds) takes considerably
longer. Results are saved after every seed, so an interrupted run keeps its partial output.

In [1]:
import sys, os
from pathlib import Path
# Resolve the project root so the notebook runs from either the repo root or notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

# Project modules: data loading and the experiment runner
from src.data import load_data
from src.run_experiments import run, build_conditions

SYNTHETIC = False   # True = fake data, ONLY to check the code runs. Never report results from it.
if SYNTHETIC:
    print("WARNING: SYNTHETIC data in use. Do not interpret or report any number below.")

In [2]:
# Load the data and build the condition grid:
# mechanism x history rate x months of transactions kept x noise level
df = load_data(synthetic=SYNTHETIC)
conds = build_conditions(quick=False)
print(len(conds), "conditions")
pd.DataFrame(conds)

32 conditions


,mechanism,hist_rate,months,noise
0,none,0.0,6,0.00
1,MCAR,0.2,6,0.00
2,MCAR,0.5,6,0.00
3,MCAR,0.8,6,0.00
4,MAR,0.2,6,0.00
5,MAR,0.5,6,0.00
6,MAR,0.8,6,0.00
7,ALL,1.0,6,0.00
8,none,0.0,6,0.25
9,none,0.0,6,0.50


## 1. Quick run (smoke test: 2 seeds, reduced condition set)

In [3]:
quick = run(seeds=2, quick=True, df=df, out=str(ROOT / "results" / "quick_results.csv"))
# Metrics recorded for every run
METRICS = ["auc", "pr_auc", "log_loss", "brier", "cal_slope", "cal_intercept", "ece", "mpd", "spearman_vs_full"]
# Stop early if any core metric is NaN
assert quick[["auc", "brier", "cal_slope", "ece"]].notna().all().all(), "NaNs in core metrics: investigate"
print("OK. Rows per regime:"); print(quick.groupby("regime").size())

30,000 rows | 13 conditions | 2 seeds | quick=True
seed 1/2 done  (155s elapsed)
seed 2/2 done  (302s elapsed)
Wrote 468 rows to C:\Users\HP\Documents\Data Analysis & Science\Proof of Work\Credit Scoring Python\credit-risk-calibration-under-scarcity\results\quick_results.csv
OK. Rows per regime:
regime
A    156
B    312
dtype: int64


## 2. Full run (5 seeds)

In [4]:
N_SEEDS = 5
# Full grid; results are written to results/results.csv after each seed
full = run(seeds=N_SEEDS, quick=False, df=df, out=str(ROOT / "results" / "results.csv"))

30,000 rows | 32 conditions | 5 seeds | quick=False
seed 1/5 done  (380s elapsed)
seed 2/5 done  (841s elapsed)
seed 3/5 done  (1365s elapsed)
seed 4/5 done  (1745s elapsed)
seed 5/5 done  (2309s elapsed)
Wrote 2,880 rows to C:\Users\HP\Documents\Data Analysis & Science\Proof of Work\Credit Scoring Python\credit-risk-calibration-under-scarcity\results\results.csv


## 3. Checks on the results file

In [5]:
# Reload from disk so the checks run on exactly what was saved
full = pd.read_csv(ROOT / "results" / "results.csv")
print("Rows:", len(full), "| seeds:", full.seed.nunique(), "| conditions:",
      full[["mechanism", "hist_rate", "months", "noise"]].drop_duplicates().shape[0])
print("NaNs in core metrics:", int(full[["auc", "brier", "cal_slope", "ece"]].isna().sum().sum()))

# Did the simulated missingness hit its target? (test set, regime A, months=6, noise=0)
chk = full[(full.regime == "A") & (full.reg == "l2") & (full.recal == "none") & (full.months == 6) & (full.noise == 0)]
chk.groupby(["mechanism", "hist_rate"])["realised_hist_missing"].agg(["mean", "std"]).round(3)

Rows: 2880 | seeds: 5 | conditions: 32
NaNs in core metrics: 0


mean    std
mechanism hist_rate              
ALL       1.0        1.000  0.000
MAR       0.2        0.202  0.007
          0.5        0.503  0.008
          0.8        0.800  0.004
MCAR      0.2        0.202  0.005
          0.5        0.500  0.005
          0.8        0.803  0.008
none      0.0        0.000  0.000

These checks confirm that the results file contains the expected seeds and conditions, that the core metrics have
no NaNs, and that the simulated missingness reached its target rates. Analysis and figures follow in notebook 04.

Runs with `SYNTHETIC = True` use generated data to test the code only; their results are not meaningful and are
not reported.